# Modelo No. 2

## 1. Importaciones y utilidades

In [23]:
import os
import cv2
import numpy as np
from datetime import datetime
import mediapipe as mp
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.regularizers import l2
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from keras.models import load_model
from mediapipe.python.solutions.holistic import Holistic
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import json

ROOT_PATH = "data"
FRAME_ACTIONS_PATH = "frases"  # carpeta por palabra/frase
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SIZE = 1
FONT_POS = (10, 30)

# Crear carpeta si no existe
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

# Detectar si hay manos en los resultados
def there_hand(results):
    return results.left_hand_landmarks or results.right_hand_landmarks

# Dibujar los keypoints de las manos
def draw_keypoints(image, results):
    if results.left_hand_landmarks:
        mp.solutions.drawing_utils.draw_landmarks(image, results.left_hand_landmarks, mp.solutions.holistic.HAND_CONNECTIONS)
    if results.right_hand_landmarks:
        mp.solutions.drawing_utils.draw_landmarks(image, results.right_hand_landmarks, mp.solutions.holistic.HAND_CONNECTIONS)

# Aplicar MediaPipe
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    return results

# Extraer keypoints de ambas manos
def extract_keypoints(results):
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21 * 3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21 * 3)
    return np.concatenate([lh, rh])

## 2. Captura de muestras

In [11]:
def capture_samples_npy(path, margin_frame=1, min_cant_frames=30, delay_frames=3):
    """
    CAPTURA DE MUESTRAS PARA UNA PALABRA Y GUARDA KEYPOINTS COMO .NPY
    - Comienza a grabar cuando detecta mano.
    - Sigue 'delay_frames' tras dejar de ver la mano, para no cortar brusco.
    - Recorta 'margin_frame' al inicio y 'delay_frames' al final ANTES de guardar.
    """
    create_folder(path)

    registro_actual = len([f for f in os.listdir(path) if f.endswith(".npy")]) + 1

    cap = cv2.VideoCapture(1)  # 0 suele ser la cámara principal (cambia si usas 1)
    window_name = "Vista previa de señas"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 800, 600)

    DISPLAY_WIDTH, DISPLAY_HEIGHT = 800, 600

    recording = False
    sequence = []
    count_frame = 0
    fix_frames = 0  # cuenta frames desde que se perdió la mano

    try:
        with mp.solutions.holistic.Holistic(min_detection_confidence=0.5,
                                            min_tracking_confidence=0.5) as holistic_model:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                image = frame.copy()
                results = mediapipe_detection(frame, holistic_model)
                hand_present = there_hand(results)

                if hand_present:
                    # Si volvimos a ver la mano, resetea el contador de "perdida"
                    fix_frames = 0

                    # Si no estaba grabando, inicia
                    if not recording:
                        recording = True
                        sequence = []
                        count_frame = 0

                    count_frame += 1

                    # Mostrar estado
                    cv2.putText(image, f'Registro {registro_actual}...', (10, 30), FONT, FONT_SIZE, (100, 200, 255), 2)
                    cv2.putText(image, 'Capturando...', (20, image.shape[0] - 20), FONT, FONT_SIZE, (0, 50, 255), 2)

                    # A partir del margen inicial, acumulamos keypoints
                    if count_frame > margin_frame:
                        keypoints = extract_keypoints(results)
                        if keypoints is not None:
                            sequence.append(keypoints)

                elif recording:
                    # Se perdió la mano pero seguimos unos frames por "delay"
                    fix_frames += 1
                    cv2.putText(image, f'Registro {registro_actual}...', (10, 30), FONT, FONT_SIZE, (100, 200, 255), 2)
                    cv2.putText(image, f'Finalizando... ({fix_frames}/{delay_frames})', (20, image.shape[0] - 20),
                                FONT, FONT_SIZE, (0, 165, 255), 2)

                    if fix_frames >= delay_frames:
                        # Recorte final: quitamos 'delay_frames' al final y 'margin_frame' (ya no agregamos durante margen, pero por seguridad)
                        if delay_frames > 0 and len(sequence) > delay_frames:
                            sequence = sequence[:-delay_frames]

                        # Validar longitud mínima
                        if len(sequence) >= min_cant_frames:
                            timestamp = datetime.now().strftime('%y%m%d%H%M%S%f')
                            npy_path = os.path.join(path, f'sample_{timestamp}.npy')
                            np.save(npy_path, np.array(sequence))
                            print(f"✅ Muestra guardada: {npy_path}")
                            registro_actual += 1
                        else:
                            print("⚠️ Muestra descartada por ser muy corta.")

                        # Reset total
                        recording = False
                        sequence = []
                        count_frame = 0
                        fix_frames = 0

                else:
                    # No se está grabando y no hay mano: estado en espera
                    cv2.putText(image, f'Registro {registro_actual}...', (10, 30), FONT, FONT_SIZE, (100, 200, 255), 2)
                    cv2.putText(image, 'Listo para capturar...', (20, image.shape[0] - 20),
                                FONT, FONT_SIZE, (0, 220, 100), 2)

                # Dibujo y display
                draw_keypoints(image, results)
                image_display = cv2.resize(image, (DISPLAY_WIDTH, DISPLAY_HEIGHT))
                cv2.imshow(window_name, image_display)

                # Salir
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    break
    finally:
        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    palabra = "donde"
    word_path = os.path.join(ROOT_PATH, FRAME_ACTIONS_PATH, palabra)
    capture_samples_npy(word_path)

## 3. Normalización

In [ ]:
ROOT_PATH = "data"
FRAME_ACTIONS_PATH = "frases"
MODEL_FRAMES = 30  

def normalize_keypoints_sequence(sequence, target_len=30):
    current_len = len(sequence)
    if current_len == target_len:
        return sequence
    elif current_len > target_len:
        # Reducir seleccionando frames uniformemente
        step = current_len / target_len
        indices = np.arange(0, current_len, step).astype(int)[:target_len]
        return sequence[indices]
    else:
        # Interpolar si es menor
        indices = np.linspace(0, current_len - 1, target_len)
        interpolated = []
        for i in indices:
            lower = int(np.floor(i))
            upper = int(np.ceil(i))
            weight = i - lower
            if upper >= current_len:
                upper = current_len - 1
            interpolated_frame = (1 - weight) * sequence[lower] + weight * sequence[upper]
            interpolated.append(interpolated_frame)
        return np.array(interpolated)

def process_npy_directory(word_directory, target_len=30):
    for fname in os.listdir(word_directory):
        if fname.endswith(".npy"):
            full_path = os.path.join(word_directory, fname)
            sequence = np.load(full_path)
            normalized = normalize_keypoints_sequence(sequence, target_len)
            np.save(full_path, normalized)
            print(f"✅ Normalizado: {fname} → {normalized.shape}")

if __name__ == "__main__":
    word_ids = [w for w in os.listdir(os.path.join(ROOT_PATH, FRAME_ACTIONS_PATH))]

    for word_id in word_ids:
        word_path = os.path.join(ROOT_PATH, FRAME_ACTIONS_PATH, word_id)
        if os.path.isdir(word_path):
            print(f"\n📦 Normalizando muestras de: {word_id}")
            process_npy_directory(word_path, target_len=MODEL_FRAMES)

# 3.1 Aumentación (solo al conjunto de entrenamiento)

In [59]:
# ===== 3) AUMENTACIÓN PARA KEYPOINTS (solo TRAIN) =====
import numpy as np
from sklearn.model_selection import train_test_split

# --- helpers para (T, F) con pares (x,y,x,y,...) ---
def _flatten_to_xy(seq):
    T, F = seq.shape
    assert F % 2 == 0, "F debe ser par (x1,y1,x2,y2,...)"
    return seq.reshape(T, F//2, 2)

def _xy_to_flat(xy):
    T, K, _ = xy.shape
    return xy.reshape(T, K*2)

def _center(xy):
    mean_xy = xy.mean(axis=(0,1), keepdims=True)
    return xy - mean_xy, mean_xy

def _restore(xy_c, mean_xy):
    return xy_c + mean_xy

def _clip01(xy):
    return np.clip(xy, 0.0, 1.0)

# --- augmentations suaves y seguras ---
def aug_jitter(seq, sigma=0.01):
    xy = _flatten_to_xy(seq)
    xy = _clip01(xy + np.random.normal(0, sigma, xy.shape))
    return _xy_to_flat(xy)

def aug_translate(seq, max_shift=0.03):
    xy = _flatten_to_xy(seq)
    shift = np.array([np.random.uniform(-max_shift, max_shift),
                      np.random.uniform(-max_shift, max_shift)])
    xy = _clip01(xy + shift)
    return _xy_to_flat(xy)

def aug_scale(seq, min_s=0.96, max_s=1.04):
    xy = _flatten_to_xy(seq)
    xy_c, m = _center(xy)
    s = np.random.uniform(min_s, max_s)
    xy = _clip01(_restore(xy_c * s, m))
    return _xy_to_flat(xy)

def aug_rotate(seq, max_deg=8):
    xy = _flatten_to_xy(seq)
    xy_c, m = _center(xy)
    th = np.deg2rad(np.random.uniform(-max_deg, max_deg))
    c, s = np.cos(th), np.sin(th)
    R = np.array([[c, -s], [s, c]])
    xy = _clip01(_restore(xy_c @ R.T, m))
    return _xy_to_flat(xy)

def aug_time_warp(seq, max_warp=0.08):
    T, F = seq.shape
    t = np.linspace(0, 1, T)
    warp = np.random.uniform(-max_warp, max_warp)
    t_src = np.clip(t + warp*(t - t**2), 0, 1)
    out = np.zeros_like(seq)
    for f in range(F):
        out[:, f] = np.interp(t, t_src, seq[:, f])
    return out

def augment_sequence(seq):
    out = seq
    if np.random.rand() < 0.7: out = aug_jitter(out, 0.01)
    if np.random.rand() < 0.5: out = aug_translate(out, 0.03)
    if np.random.rand() < 0.5: out = aug_scale(out, 0.96, 1.04)
    if np.random.rand() < 0.5: out = aug_rotate(out, 8)
    if np.random.rand() < 0.5: out = aug_time_warp(out, 0.08)
    return out

def augment_batch(X, times=1):
    """
    X: (N,T,F) -> devuelve (times*N, T, F) con variaciones.
    times=1 duplica el train; times=2 triplica, etc.
    """
    N, T, F = X.shape
    outs = []
    for _ in range(times):
        Xa = np.empty_like(X)
        for i in range(N):
            Xa[i] = augment_sequence(X[i])
        outs.append(Xa)
    return np.concatenate(outs, axis=0)

# ---- NORMALIZAR (estándar) + AUMENTAR SOLO TRAIN ----
def prepare_with_augmentation(X, y, test_size=0.2, seed=42, aug_times=1):
    # Split estratificado igual que ya usas
    X_train, X_val, y_train, y_val = train_test_split(
        X.astype(np.float32), y,
        test_size=test_size, random_state=seed,
        stratify=np.argmax(y, axis=1)
    )

    # Normalización por feature (media/std) usando SOLO TRAIN
    mu = X_train.mean(axis=(0,1), keepdims=True)             # (1,1,F)
    sigma = X_train.std(axis=(0,1), keepdims=True) + 1e-6    # (1,1,F)

    X_train_n = (X_train - mu) / sigma
    X_val_n   = (X_val   - mu) / sigma  # mismo mu/sigma

    # Aumentación SOLO al train (después de normalizar)
    if aug_times > 0:
        X_aug = augment_batch(X_train_n, times=aug_times)
        y_aug = np.concatenate([y_train.copy() for _ in range(aug_times)], axis=0)
        X_train_final = np.concatenate([X_train_n, X_aug], axis=0)
        y_train_final = np.concatenate([y_train,   y_aug], axis=0)
    else:
        X_train_final, y_train_final = X_train_n, y_train

    return X_train_final, y_train_final, X_val_n, y_val, mu, sigma


## 4. Entrenamiento

In [60]:
# Número total de keypoints por frame: 21 puntos × 3 coords × 2 manos
# Ajusta estas rutas a tu proyecto
MODEL_PATH = "models/action.h5"

LENGTH_KEYPOINTS = 126      # SOLO manos (21*3*2)
MODEL_FRAMES = 30           # frames por secuencia (coincidir con inferencia)

def load_dataset(base_path="data/frases"):
    X = []
    y = []
    label_map = {}

    palabras = sorted(os.listdir(base_path))
    for idx, palabra in enumerate(palabras):
        label_map[palabra] = idx
        palabra_path = os.path.join(base_path, palabra)
        for archivo in os.listdir(palabra_path):
            if archivo.endswith(".npy"):
                secuencia = np.load(os.path.join(palabra_path, archivo))
                if secuencia.shape[0] == MODEL_FRAMES:
                    X.append(secuencia)
                    y.append(idx)

    X = np.array(X)
    y = to_categorical(np.array(y))
    return X, y, label_map

In [61]:
def get_model(max_length_frames, output_length: int):
    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=(max_length_frames, LENGTH_KEYPOINTS), kernel_regularizer=l2(0.01)))
    model.add(Dropout(0.5))
    model.add(LSTM(128, return_sequences=False, kernel_regularizer=l2(0.001)))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dense(output_length, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
import os, random, numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, TerminateOnNaN, ModelCheckpoint
import matplotlib.pyplot as plt

# --- Semilla y memoria GPU (estables, no cambian tu flujo) ---
def _set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)

def train_model_from_npy(epochs=300):
    _set_seed(42)

    # Carga
    X, y, label_map = load_dataset()  # y one-hot con num_clases = len(label_map)
    print(f"✅ Total muestras: {X.shape[0]}, Clases: {len(label_map)}")

    # Sanity checks
    assert X.ndim == 3 and X.shape[1] == MODEL_FRAMES and X.shape[2] == LENGTH_KEYPOINTS, \
        f"X shape {X.shape} debe ser (N, {MODEL_FRAMES}, {LENGTH_KEYPOINTS})"
    assert y.ndim == 2, f"y debe ser one-hot (N, C); shape actual: {y.shape}"

    X = X.astype(np.float32)

    # Split estratificado
    X_train, y_train, X_val, y_val, mu, sigma = prepare_with_augmentation(
        X, y, test_size=0.2, seed=42, aug_times=1   # aug_times=1 duplica el train
    )

    # Modelo
    model = get_model(MODEL_FRAMES, output_length=y.shape[1])
    print("MODEL in/out (entrenamiento):", model.input_shape, "->", model.output_shape)

    # Callbacks sólidos (no cambian tu lógica)
    early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1)
    reduce_lr  = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)
    nan_killer = TerminateOnNaN()
    ckpt = ModelCheckpoint(filepath="best_model.h5", monitor="val_loss",
                           save_best_only=True, save_weights_only=False, verbose=1)
    callbacks = [early_stop, reduce_lr, nan_killer, ckpt]

    # Pesos por clase (robusto si hay desbalance; no rompe si todo está balanceado)
    from sklearn.utils.class_weight import compute_class_weight
    y_train_idx = np.argmax(y_train, axis=1)
    classes = np.arange(y.shape[1])
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_idx)
    class_weights = {i: w for i, w in enumerate(weights)}
    print("⚖️ class_weights:", class_weights)

    # Entrenamiento
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=8,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1
    )

    # Guardar modelo completo
    model.save(MODEL_PATH)
    print(f"💾 Modelo guardado en: {MODEL_PATH}")

    # --- Evaluación y visualización ---
    # Alinear etiquetas por índice (evita desorden con dict.keys())
    idx_to_label = {v: k for k, v in label_map.items()}
    display_labels = [idx_to_label[i] for i in range(len(idx_to_label))]

    y_pred = model.predict(X_val)
    y_pred_labels = np.argmax(y_pred, axis=1)
    y_true = np.argmax(y_val, axis=1)

    cm = confusion_matrix(y_true, y_pred_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
    disp.plot(cmap="Blues", values_format=".0f")
    plt.title("Matriz de Confusión - Validación")
    plt.show()

    # Reporte por clase (precisión/recall/F1) – solo imprime
    print("\n", classification_report(y_true, y_pred_labels, target_names=display_labels, digits=3))

    # Diagnóstico rápido (sin guardar historial)
    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    print(f"\n📊 Accuracy final - Entrenamiento: {train_acc:.2f}, Validación: {val_acc:.2f}")
    if abs(train_acc - val_acc) > 0.2:
        print("⚠️ Posible overfitting")
    elif max(history.history['val_accuracy']) < 0.6:
        print("⚠️ Posible underfitting")
    else:
        print("✅ Buen desempeño general")

    # Eval del mejor checkpoint (opcional pero útil, no cambia nada del flujo)
    try:
        best = tf.keras.models.load_model("best_model.h5")
        best_loss, best_acc = best.evaluate(X_val, y_val, verbose=0)
        print(f"🏁 Mejor checkpoint -> loss: {best_loss:.4f}, acc: {best_acc:.4f}")
    except Exception as e:
        print("ℹ️ No se pudo evaluar best_model.h5:", e)

    return model, label_map

# Ejecución (igual que tu ejemplo)
model, label_map = train_model_from_npy(epochs=150)

✅ Total muestras: 3600, Clases: 9
MODEL in/out (entrenamiento): (None, 30, 126) -> (None, 9)
⚖️ class_weights: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0}
Epoch 1/150
683/720 [===========================>..] - ETA: 1s - loss: 1.4037 - accuracy: 0.7141

## 5. Validaciones

In [51]:
LENGTH_KEYPOINTS = 126

def get_model(max_length_frames, output_length: int):
    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=(max_length_frames, LENGTH_KEYPOINTS), kernel_regularizer=l2(0.01)))
    model.add(Dropout(0.5))
    model.add(LSTM(128, return_sequences=False, kernel_regularizer=l2(0.001)))
    model.add(Dropout(0.5))
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dense(output_length, activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [52]:
MODEL_PATH = "models/action.h5"
DATA_PATH = "data/frases"
WORDS = sorted(os.listdir(DATA_PATH))  # nombres de las clases
MODEL_FRAMES = 30
MIN_LENGTH_FRAMES = 10
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_POS = (10, 30)
FONT_SIZE = 1

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    return model.process(image)

def there_hand(results):
    return results.left_hand_landmarks or results.right_hand_landmarks

def extract_keypoints(results):
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]) if results.left_hand_landmarks else np.zeros((21, 3))
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]) if results.right_hand_landmarks else np.zeros((21, 3))
    return np.concatenate([lh, rh]).flatten()

def normalize_keypoints(keypoints, target_length=30):
    current_length = len(keypoints)
    if current_length == target_length:
        return keypoints
    elif current_length > target_length:
        step = current_length / target_length
        indices = np.arange(0, current_length, step).astype(int)[:target_length]
        return [keypoints[i] for i in indices]
    else:
        indices = np.linspace(0, current_length - 1, target_length)
        interpolated = []
        for i in indices:
            lower = int(np.floor(i))
            upper = int(np.ceil(i))
            weight = i - lower
            if lower == upper:
                interpolated.append(keypoints[lower])
            else:
                interpolated.append(((1 - weight) * np.array(keypoints[lower]) + weight * np.array(keypoints[upper])).tolist())
        return interpolated

def draw_keypoints(image, results):
    mp_drawing = mp.solutions.drawing_utils
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp.solutions.holistic.HAND_CONNECTIONS)
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp.solutions.holistic.HAND_CONNECTIONS)

In [55]:
def evaluate_model(threshold=0.8, margin_frame=1, delay_frames=3, src=0):
    print("🟡 Cargando modelo...")
    model = load_model(MODEL_PATH)

    print("📷 Inicializando cámara...")
    cap = cv2.VideoCapture(src)
    if not cap.isOpened():
        print("❌ No se pudo abrir la cámara.")
        return

    sentence = []
    pred_label = ""
    pred_prob = 0.0
    kp_seq, count_frame, fix_frames = [], 0, 0
    recording = False

    with Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        print("🎥 Traductor en vivo iniciado. Presiona 'q' para salir.")
        while True:
            ret, frame = cap.read()
            if not ret:
                print("⚠️ No se pudo leer el frame de la cámara.")
                break

            results = mediapipe_detection(frame, holistic)

            if there_hand(results) or recording:
                recording = False
                count_frame += 1
                if count_frame > margin_frame:
                    kp_frame = extract_keypoints(results)
                    kp_seq.append(kp_frame)
            else:
                if count_frame >= MIN_LENGTH_FRAMES + margin_frame:
                    fix_frames += 1
                    if fix_frames < delay_frames:
                        recording = True
                        continue

                    kp_seq = kp_seq[: -(margin_frame + delay_frames)]
                    kp_normalized = normalize_keypoints(kp_seq, target_length=MODEL_FRAMES)
                    input_data = np.expand_dims(np.array(kp_normalized), axis=0)

                    prediction = model.predict(input_data, verbose=0)[0]
                    max_idx = np.argmax(prediction)
                    max_prob = prediction[max_idx]

                    if max_prob > threshold:
                        pred_label = WORDS[max_idx]
                        pred_prob = max_prob * 100
                        sentence.insert(0, pred_label)

                kp_seq = []
                count_frame = 0
                fix_frames = 0
                recording = False

            # Mostrar texto y porcentaje en ventana
            cv2.rectangle(frame, (0, 0), (640, 60), (245, 117, 16), -1)
            cv2.putText(frame, f'{pred_label.upper()} ({pred_prob:.1f}%)', FONT_POS, FONT, FONT_SIZE, (255, 255, 255), 2)

            draw_keypoints(frame, results)
            cv2.imshow('Traductor LSP', frame)

            if cv2.waitKey(10) & 0xFF == ord('q'):
                print("🛑 Cerrando traductor.")
                break

        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    evaluate_model()

🟡 Cargando modelo...
📷 Inicializando cámara...
🎥 Traductor en vivo iniciado. Presiona 'q' para salir.
🛑 Cerrando traductor.


In [47]:
from contextlib import contextmanager
def evaluate_video(
    video_path: str,
    threshold: float = 0.8,
    margin_frame: int = 1,
    delay_frames: int = 3,
    output: bool = True,
    show_window: bool = False,
):
    """
    Evalúa un video en archivo en lugar de la cámara.
    - Detecta ventanas de gesto (aparece mano -> desaparece) y predice con el modelo.
    - Genera un MP4 anotado con el texto (si output=True).
    - Devuelve una lista de detecciones con tiempos de inicio/fin y probabilidad.
    """
    if not os.path.isfile(video_path):
        print(f"❌ No existe el archivo: {video_path}")
        return []

    print("🟡 Cargando modelo...")
    model = load_model(MODEL_PATH)

    print(f"🎬 Abriendo video: {video_path}")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ No se pudo abrir el video.")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)  or 640)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 480)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    # Preparar salida de video
    writer = None
    out_path = None
    if output:
        base = os.path.splitext(os.path.basename(video_path))[0]
        out_path = f"resultado_{base}.mp4"
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    detections = []  # cada item: {label, prob, start_frame, end_frame, start_time, end_time}

    kp_seq, count_frame, fix_frames = [], 0, 0
    recording = False

    pred_label = ""
    pred_prob = 0.0
    overlay_cooldown = 0  # frames para mostrar el texto en el video

    # --- MediaPipe Holistic context manager (usa tu helper si lo tienes) ---
    import mediapipe as mp
    with mp.solutions.holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        frame_idx = 0
        print("▶️ Procesando frames...")
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Detección con tu helper (si ya lo tienes). Si no, usa el pipeline normal de MP:
            # results = mediapipe_detection(frame, holistic)
            # Si tu mediapipe_detection ya convierte a RGB y llama holistic.process, úsalo.
            # Aquí uso el helper por compatibilidad con tu notebook:
            results = mediapipe_detection(frame, holistic)

            if there_hand(results) or recording:
                recording = False
                count_frame += 1
                if count_frame > margin_frame:
                    kp_frame = extract_keypoints(results)  # Debe regresar vector de 126 (o lo que uses)
                    kp_seq.append(kp_frame)
            else:
                # Sección de cierre del gesto
                if count_frame >= MIN_LENGTH_FRAMES + margin_frame:
                    fix_frames += 1
                    if fix_frames < delay_frames:
                        recording = True
                        # seguimos acumulando el "colchón" de cierre
                    else:
                        # recorta margen + delay
                        if margin_frame + delay_frames > 0:
                            kp_seq = kp_seq[:-(margin_frame + delay_frames)] if len(kp_seq) > (margin_frame + delay_frames) else kp_seq

                        # normaliza y remuestrea a MODEL_FRAMES
                        kp_normalized = normalize_keypoints(kp_seq, target_length=MODEL_FRAMES)  # (30, D)
                        input_data = np.expand_dims(np.array(kp_normalized), axis=0)            # (1,30,D)

                        prediction = model.predict(input_data, verbose=0)[0]
                        max_idx = int(np.argmax(prediction))
                        max_prob = float(prediction[max_idx])

                        if max_prob > threshold:
                            pred_label = WORDS[max_idx]
                            pred_prob = max_prob * 100.0
                            overlay_cooldown = int(fps * 0.7)  # muestra ~0.7s el texto

                            # marca temporal (aprox): usamos frame actual como fin, y restamos la duración de la ventana
                            end_frame = frame_idx
                            dur_frames = len(kp_seq)
                            start_frame = max(0, end_frame - dur_frames)
                            detections.append({
                                "label": pred_label,
                                "prob": pred_prob,
                                "start_frame": start_frame,
                                "end_frame": end_frame,
                                "start_time": start_frame / fps,
                                "end_time": end_frame / fps,
                            })

                        # reset para siguiente gesto
                        kp_seq, count_frame, fix_frames, recording = [], 0, 0, False
                else:
                    # No hubo suficiente material para formar gesto
                    kp_seq, count_frame, fix_frames, recording = [], 0, 0, False

            # --- Overlay de texto (si hubo predicción reciente) ---
            if overlay_cooldown > 0:
                cv2.rectangle(frame, (0, 0), (max(300, width // 2), 60), (245, 117, 16), -1)
                cv2.putText(frame, f'{pred_label.upper()} ({pred_prob:.1f}%)',
                            (20, 40), FONT, FONT_SIZE, (255, 255, 255), 2)
                overlay_cooldown -= 1

            # Dibuja keypoints si quieres mantener tu visualización
            try:
                draw_keypoints(frame, results)
            except Exception:
                pass  # por si tu helper no está cargado aquí

            # Escribir/frame y/o mostrar
            if writer is not None:
                writer.write(frame)

            if show_window:
                cv2.imshow('Traductor LSP (video)', frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            frame_idx += 1

    cap.release()
    if writer is not None:
        writer.release()
    if show_window:
        cv2.destroyAllWindows()

    # --- Resumen ---
    print("✅ Detecciones:")
    if not detections:
        print("   (sin resultados sobre el umbral)")
    else:
        for i, d in enumerate(detections, 1):
            print(f" {i:02d}. {d['label']}  {d['prob']:.1f}%  "
                  f"{d['start_time']:.2f}s → {d['end_time']:.2f}s  "
                  f"(frames {d['start_frame']}–{d['end_frame']})")

    if out_path:
        print(f"💾 Video anotado: {out_path}")

    return detections

In [48]:
detecciones = evaluate_video(
    "prueba.mp4",   # ruta a tu video
    threshold=0.8,   # probabilidad mínima aceptada
    output=True,     # genera un video anotado
    show_window=False # pon True si quieres que se abra una ventana en vivo
)

print(detecciones)

🟡 Cargando modelo...
🎬 Abriendo video: prueba.mp4
▶️ Procesando frames...
✅ Detecciones:
 01. gracias  97.8%  0.80s → 2.28s  (frames 48–137)
 02. gracias  99.9%  3.63s → 5.03s  (frames 218–302)
 03. donde  97.4%  7.80s → 9.62s  (frames 468–577)
 04. donde  99.2%  11.00s → 12.63s  (frames 660–758)
 05. cual  100.0%  14.85s → 16.88s  (frames 891–1013)
 06. cual  100.0%  18.02s → 20.05s  (frames 1081–1203)
💾 Video anotado: resultado_prueba.mp4
[{'label': 'gracias', 'prob': 97.76325225830078, 'start_frame': 48, 'end_frame': 137, 'start_time': 0.8, 'end_time': 2.283333333333333}, {'label': 'gracias', 'prob': 99.9140977859497, 'start_frame': 218, 'end_frame': 302, 'start_time': 3.6333333333333333, 'end_time': 5.033333333333333}, {'label': 'donde', 'prob': 97.37966060638428, 'start_frame': 468, 'end_frame': 577, 'start_time': 7.8, 'end_time': 9.616666666666667}, {'label': 'donde', 'prob': 99.17045831680298, 'start_frame': 660, 'end_frame': 758, 'start_time': 11.0, 'end_time': 12.6333333333333